# Lab 3 — Auto Loader Incremental Ingestion


This notebook demonstrates incremental ingestion of synthetic wind turbine telemetry files from ADLS into a Bronze Delta table using Databricks Auto Loader and Structured Streaming.

## 1. Pipeline Parameters

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("source_path", "")
dbutils.widgets.text("schema_path", "")
dbutils.widgets.text("checkpoint_path", "")
dbutils.widgets.text("target_table", "turbine_telemetry")

In [0]:
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
source_path = dbutils.widgets.get("source_path")
schema_path = dbutils.widgets.get("schema_path")
checkpoint_path = dbutils.widgets.get("checkpoint_path")
target_table = dbutils.widgets.get("target_table")

full_table_name = f"{catalog}.{bronze_schema}.{target_table}"

## 2. Auto Loader Source

In [0]:
df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(source_path)
)

## 3. Bronze Metadata

In [0]:
from pyspark.sql import functions as F
df_bronze = (
 df.withColumn("source_filename", F.col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

## 4. Incremental Write to Bronze

In [0]:
query = (
    df_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(full_table_name)
)

query.awaitTermination()

## Incremental Ingestion

The source files were uploaded incrementally in several batches rather than all at once.

- Initial batch: 400 files
- Second batch: 200 files
- Third batch: 200 files
- Fourth batch: 200 files

After the first 400 source files were processed, the Bronze table contained 4,000 records.

![image_1789673798842.png](./image_1789673798842.png "image_1789673798842.png")

### Final ingestion result

After all 1,000 base files and 10 additional schema-evolution files were processed, the Bronze table contained 10,100 records.

In [0]:
row_count = spark.table(full_table_name).count()
print(f"Rows in Bronze: {row_count}")


## Schema Evolution Experiment

A new 'maintenance_required' column was introduced in additional source files.
On the first processing attempt, Auto Loader detected the previously unknown field and updated the tracked schema. The streaming query required a restart, as shown below.

![image_1789674180047.png](./image_1789674180047.png "image_1789674180047.png")

After restarting the stream with the same schema and checkpoint locations, ingestion completed successfully and the new column became part of the Bronze table schema. Records originating from the previous schema contain 'NULL' for this column.

In [0]:
spark.table(full_table_name).printSchema()

### Rescued Data Check


In [0]:
spark.table(full_table_name) \
    .select("_rescued_data") \
    .where(F.col("_rescued_data").isNotNull()) \
    .show(truncate=False)

## Safe Reload and Idempotency Check

The pipeline was executed again with the same source and checkpoint location. Since all available files had already been processed, no additional records were written to the Bronze table.

In [0]:
before_count = spark.table(full_table_name).count()

reload_query = (
    df_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(full_table_name)
)

reload_query.awaitTermination()

after_count = spark.table(full_table_name).count()

print(f"Before: {before_count}")
print(f"After:  {after_count}")

## Trigger and Streaming Statistics Experiment

In [0]:

once_checkpoint_path = f"{checkpoint_path}_once_test"
once_table = f"{catalog}.{bronze_schema}.turbine_telemetry_once_test"

In [0]:
df_once = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", f"{schema_path}_once_test")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.maxFilesPerTrigger", 300)
        .load(source_path)
)

In [0]:
df_once_bronze = (
    df_once
        .withColumn("source_filename", F.col("_metadata.file_name"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("load_date", F.current_date())
)

In [0]:
once_query = (
    df_once_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", once_checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(once=True)
        .toTable(once_table)
)

once_query.awaitTermination()

In [0]:
print("Rows processed:", spark.table(once_table).count())
once_query.recentProgress

The test stream was executed using the 'once' trigger with a separate checkpoint and target table. Since the checkpoint was new, Auto Loader treated the source files as unprocessed.

The query processed 10,100 records in a single micro-batch ('batchId = 0') and then stopped. No files remained outstanding.

The main ingestion pipeline uses 'availableNow', while this isolated experiment demonstrates the behavior of the 'once' trigger without modifying the main Bronze table or its checkpoint.

## Checkpoint Management Experiment

The main pipeline checkpoint is preserved. A separate test stream is used to demonstrate how checkpoint state affects reprocessing.

First, the stream is restarted with its existing checkpoint. Then the test checkpoint is reset to demonstrate that losing checkpoint state can cause previously processed source data to be processed again.

In [0]:
before_count = spark.table(once_table).count()

same_checkpoint_query = (
    df_once_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", once_checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(once=True)
        .toTable(once_table)
)

same_checkpoint_query.awaitTermination()

after_count = spark.table(once_table).count()

print(f"Before restart: {before_count}")
print(f"After restart:  {after_count}")

In [0]:
dbutils.fs.rm(once_checkpoint_path, True)
#Test checkpoint removed

before_reset_reload = spark.table(once_table).count()

reset_query = (
    df_once_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", once_checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(once=True)
        .toTable(once_table)
)

reset_query.awaitTermination()

after_reset_reload = spark.table(once_table).count()

print(f"Before checkpoint reset reload: {before_reset_reload}")
print(f"After checkpoint reset reload:  {after_reset_reload}")


## Streaming Statistics 


In [0]:
stats_table = f"{catalog}.{bronze_schema}.turbine_telemetry_stats_test"

stats_schema_path = f"{schema_path}_stats_test"
stats_checkpoint_path = f"{checkpoint_path}_stats_test"

In [0]:
df_stats = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", stats_schema_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.maxFilesPerTrigger", 300)
        .load(source_path)
)

In [0]:
df_stats_bronze = (
    df_stats
        .withColumn("source_filename", F.col("_metadata.file_name"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("load_date", F.current_date())
)

In [0]:
stats_query = (
    df_stats_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", stats_checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(stats_table)
)

stats_query.awaitTermination()

In [0]:
stats_query.recentProgress

In [0]:
for progress in stats_query.recentProgress:
    source = progress["sources"][0]

    print(
        f"Batch {progress['batchId']}: "
        f"input rows = {progress['numInputRows']}, "
        f"files outstanding = {source['metrics'].get('numFilesOutstanding', 'N/A')}"
    )

The Databricks Spark UI reported four completed micro-batches. This demonstrates how Auto Loader can process a large set of available files incrementally across multiple micro-batches.

The experiment was isolated from the main Bronze pipeline, so the main table and checkpoint were not modified.

![image_1789678233931.png](./image_1789678233931.png "image_1789678233931.png")